# Epidemics on a contact network

*others · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/others/network_sir)

SIR and SIS epidemics on a real face-to-face contact network, with hundreds of stochastic outbreaks run at once as one batched tensor and the individual-based mean-field ODE beside them. An outbreak can only take off above the threshold $\beta/\gamma > 1/\lambda_{\max}(A)$ set by the largest eigenvalue of the adjacency matrix, and on a scale-free network vaccinating the hubs stops it with far fewer doses than vaccinating at random.

**Vector form**

$$\dot s_i = -\beta\, s_i \sum_j A_{ij} x_j,\qquad \dot x_i = \beta\, s_i \sum_j A_{ij} x_j - \gamma\, x_i,\qquad \dot r_i = \gamma\, x_i,\qquad \text{outbreak} \iff \frac{\beta}{\gamma} > \frac{1}{\lambda_{\max}(A)}$$

**Index (tensor) form**

$$P_{bi} = 1 - e^{-\beta_b \Delta t\, A_{ij} I_{bj}} \qquad \dot x_{bi} = \beta_b\, s_{bi} A_{ij} x_{bj} - \gamma\, x_{bi} \qquad z = 1 - e^{-R_0 z}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $A_{ij}$ | `load_workplace(device)` | (N, N) | 1 if people $i$ and $j$ were seen face to face, else 0 (symmetric, dense) |
| $I_{bj}$ | `state == I` | (B, N) | who is infected in each of the $B$ realisations run together |
| $A_{ij} I_{bj}$ | `einsum('ij,bj->bi', A, infected)` | (B, N) | repeated $j$ ⇒ summed: the number of infected contacts of each person |
| $P_{bi}$ | `1 - exp(-beta[:, None] * dt * pressure)` | (B, N) | chance a susceptible is infected in one step $\Delta t$; infected recover with chance $1 - e^{-\gamma \Delta t}$ |
| $s_{bi},\ x_{bi}$ | `mean_field(A, beta, gamma, x0, dt, n_steps)` | (T, 3, B, N) | probabilities that person $i$ is susceptible, infected or recovered, integrated with RK4 |
| $\lambda_{\max}$ | `eigvalsh(A)[-1]` | () | largest eigenvalue: the linearised mean field $\dot x = (\beta A - \gamma)x$ grows only if $\beta \lambda_{\max} > \gamma$ |
| $z$ | `final_size(r0)` | (R,) | well-mixed final size, the root of $z = 1 - e^{-R_0 z}$ (Kermack–McKendrick) |

Each person is S, I or R; in every step $\Delta t$ a susceptible catches the infection from each infected contact at rate $\beta$ and an infected person recovers at rate $\gamma$ (in SIS they become susceptible again). The tests check that people are only moved between compartments, that a $k$-regular graph has $\lambda_{\max} = k$, that below the threshold outbreaks fizzle, and that on a complete graph (everybody meets everybody, $R_0 = \beta(N-1)/\gamma$) the mean final size of major outbreaks matches the Kermack–McKendrick (1927) equation $z = 1 - e^{-R_0 z}$ ($z \approx 0.80$ for $R_0 = 2$). The threshold $1/\lambda_{\max}$ comes from linearising the individual-based mean field; it is exact for that ODE, but the ODE treats neighbours as independent and lets infection echo back and forth along the same edge, so on this small, dense, clustered network it overestimates how big the epidemic gets. An SIR infection cannot travel straight back along the edge it came from: the message-passing threshold $T = \beta/(\beta + \gamma) = 1/\lambda_{NB}$, with $\lambda_{NB}$ the leading eigenvalue of the non-backtracking matrix (Karrer, Newman and Zdeborová 2014), is drawn too. The simulated curve is the mean over all runs, including the many that die out by chance after the first case, so it rises gradually. On scale-free networks $\lambda_{\max}$ grows with the biggest hub, so the threshold is tiny (Pastor-Satorras and Vespignani 2001), and removing the hubs lowers $\lambda_{\max}$ fast: herd immunity with far fewer doses. The Erdős–Rényi network with the same mean degree has no big hubs, so targeting helps much less. Data: SocioPatterns "Contacts in a workplace" (office building in France, 2013; 92 people, 755 pairs seen face to face in 20-second windows, aggregated here), M. Génois et al., Network Science 3, 326 (2015), released under a Creative Commons Public Domain Dedication (CC0), www.sociopatterns.org. The BA and ER networks are synthetic. Simulated people and rates are illustrative, not a model of any real disease.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/others/network_sir/main.py
"""Main loop: real contact network → batched stochastic SIR + mean field → threshold sweep → vaccination → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import equation, plot, solve, tensors

GAMMA, DT = 1.0, 0.1  # recovery rate (time unit = the mean infectious period) and time step


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    gen = torch.Generator().manual_seed(0)
    runs = 20 if fast else params["n_runs"]
    n_steps = 120 if fast else 400

    A = tensors.load_workplace(dev)
    n = A.shape[0]
    deg = A.sum(1)
    lam, lam_nb = equation.lambda_max(A).item(), equation.lambda_nonbacktracking(A).item()
    print(f"workplace network: N = {n} people, {int(deg.sum()) // 2} contact pairs, mean degree {deg.mean():.1f}, "
          f"max {int(deg.max())}")  # fmt: skip
    print(f"λ_max(A) = {lam:.2f} → mean-field threshold β/γ = 1/λ_max = {1 / lam:.3f}")
    print(f"non-backtracking λ_NB = {lam_nb:.2f} → message-passing SIR threshold β/γ = {1 / (lam_nb - 1):.3f}")

    # one epidemic, B realisations at once; realisation 0 starts at the biggest hub and is drawn
    ratio = params["beta_x_lambda"] / lam  # β/γ
    beta = torch.full((runs,), ratio * GAMMA, device=dev)
    first = torch.randint(n, (runs,), generator=gen).to(dev)
    first[0] = int(deg.argmax())
    infected0 = torch.zeros(runs, n, dtype=torch.bool, device=dev)
    infected0[torch.arange(runs), first] = True
    sim = solve.simulate(A, beta, GAMMA, DT, n_steps, infected0, None, gen, record=True)
    x0 = torch.zeros(1, n, dtype=torch.float64, device=dev)
    x0[0, first[0]] = 1.0
    mf = solve.mean_field(A, beta[:1], GAMMA, x0, DT, n_steps)[:, :, 0].sum(-1)  # (T, 3) expected S, I, R
    conserved = bool((sim["counts"].sum(-1) == n).all())
    final = sim["ever"][-1].float() / n
    print(f"β/γ = {ratio:.3f} ({params['beta_x_lambda']:.1f} × the mean-field threshold), {runs} realisations: "
          f"S + I + R = N at every step: {conserved}")  # fmt: skip
    print(f"  fraction ever infected: mean {final.mean():.2f}, realisations with a major outbreak (> 20 %) "
          f"{(final > 0.2).float().mean():.0%}; mean field {mf[-1, 2] / n:.2f}")  # fmt: skip

    # the threshold: final size against β/γ (all ratios × runs in one batch)
    ratios = torch.linspace(0, 4, 9 if fast else 25, dtype=torch.float64) / lam
    sweep_sim, sweep_mf = solve.threshold_sweep(A, ratios.to(dev), GAMMA, DT, n_steps, 10 if fast else 80, gen)
    for k in range(0, len(ratios), max(1, len(ratios) // 6)):
        print(f"  β/γ · λ_max = {ratios[k] * lam:4.2f}: simulated final size {sweep_sim[k]:.3f}, mean field "
              f"{sweep_mf[k]:.3f}")  # fmt: skip

    # vaccination on synthetic networks with the same number of people and links
    n_syn = 200 if fast else params["n_synthetic"]
    ba = tensors.barabasi_albert(n_syn, 3, gen, dev)
    er = tensors.erdos_renyi(n_syn, ba.sum(1).mean().item(), gen, dev)
    fractions = torch.linspace(0, 0.4, 5 if fast else 9)
    vacc = {}
    for name, G in (("Barabási–Albert", ba), ("Erdős–Rényi", er)):
        vacc[name] = solve.vaccination_sweep(G, fractions, params["vacc_beta"], GAMMA, DT, n_steps,
                                             8 if fast else 30, gen)  # fmt: skip
        lam_g = equation.lambda_max(G).item()
        t, r = vacc[name]["targeted"], vacc[name]["random"]
        print(f"{name}: mean degree {G.sum(1).mean():.1f}, max {int(G.sum(1).max())}, λ_max = {lam_g:.1f}; "
              f"vaccinate {fractions[2]:.0%}: hubs → {t[0][2]:.2f} infected, random → {r[0][2]:.2f}")  # fmt: skip
    pos = tensors.layout(A)
    return plot.animate(A, pos, sim, mf, lam, lam_nb, ratio, ratios, sweep_sim, sweep_mf, fractions, vacc,
                        params["vacc_beta"], DT, out_dir / "network_sir.gif")  # fmt: skip


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/others/network_sir/equation.py
r"""Epidemics on a network → tensor translation.

Indices: b = realisation (B run at once, each with its own β_b), i, j = people (nodes).

Stochastic SIR in small time steps Δt (one coin per person per step):

    \text{pressure}_{bi} = A_{ij} I_{bj}                                  (j summed: infected contacts of i)
    P(S \to I)_{bi} = 1 - e^{-\beta_b \Delta t\, \text{pressure}_{bi}},    P(I \to R) = 1 - e^{-\gamma \Delta t}

For SIS the recovered go straight back to S. Individual-based mean field (each person infected independently with
probability x_i, susceptible with probability s_i):

    \dot s_{bi} = -\beta_b\, s_{bi} A_{ij} x_{bj},   \dot x_{bi} = \beta_b\, s_{bi} A_{ij} x_{bj} - \gamma x_{bi},
    \dot r_{bi} = \gamma x_{bi}

Near the disease-free state \dot x = (\beta A - \gamma)\,x, which grows iff β/γ > 1/λ_max(A): the epidemic threshold.
Well mixed (A = all ones), with R_0 = β(N−1)/γ, the fraction ever infected z solves z = 1 − e^{−R_0 z}.
"""

import torch


def pressure(A: torch.Tensor, infected: torch.Tensor) -> torch.Tensor:
    """pressure_{bi} = A_{ij} I_{bj}: the number of infected contacts of each person.  j summed."""
    return torch.einsum("ij,bj->bi", A, infected.to(A.dtype))


def infection_prob(beta: torch.Tensor, dt: float, press: torch.Tensor) -> torch.Tensor:
    """P(S → I in Δt) = 1 − exp(−β_b Δt pressure_{bi}): independent chances from each infected contact."""
    return 1.0 - torch.exp(-beta[:, None] * dt * press)


def mean_field_rhs(y: torch.Tensor, A: torch.Tensor, beta: torch.Tensor, gamma: float) -> torch.Tensor:
    """d/dt of y = (s, x, r) stacked as (3, B, N).  A x is the einsum 'ij,bj->bi' (j summed)."""
    s, x, _ = y
    force = beta[:, None] * s * torch.einsum("ij,bj->bi", A.to(y.dtype), x)
    return torch.stack([-force, force - gamma * x, gamma * x])


def lambda_max(A: torch.Tensor) -> torch.Tensor:
    """Largest eigenvalue of the symmetric adjacency matrix (works on stacks (..., N, N))."""
    return torch.linalg.eigvalsh(A.double())[..., -1]


def lambda_nonbacktracking(A: torch.Tensor) -> torch.Tensor:
    """Leading eigenvalue of the non-backtracking (Hashimoto) matrix, via the Ihara–Bass 2N × 2N matrix
    [[A, I − D], [I, 0]], D = diag(degrees). Message passing puts the SIR threshold at transmissibility 1/λ_NB."""
    n = A.shape[0]
    A = A.double()
    eye = torch.eye(n, dtype=A.dtype, device=A.device)
    M = torch.cat([torch.cat([A, eye - torch.diag(A.sum(1))], 1), torch.cat([eye, torch.zeros_like(A)], 1)], 0)
    return torch.linalg.eigvals(M).real.max()


def final_size(r0: torch.Tensor, n_iter: int = 2000) -> torch.Tensor:
    """Positive root of z = 1 − exp(−R0 z) for every R0 (batched); 0 when R0 ≤ 1. Fixed-point iteration from z = 1."""
    z = torch.ones_like(r0)
    for _ in range(n_iter):
        z = 1.0 - torch.exp(-r0 * z)
    return torch.where(r0 > 1, z, torch.zeros_like(z))


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/others/network_sir/tensors.py
"""Build the concrete tensors: contact networks as dense adjacency matrices A_{ij} (N, N), symmetric 0/1.

- load_workplace: a real face-to-face contact network (SocioPatterns, office building in France, 2013; CC0)
- erdos_renyi: every pair linked independently with the same probability (a Poisson degree distribution)
- barabasi_albert: preferential attachment, a scale-free network with a few big hubs (P(k) ∝ k^-3)
- complete_graph, ring_lattice: the well-mixed limit and a k-regular test case
"""

from pathlib import Path

import torch

DATA = Path(__file__).parent / "data" / "workplace_invs13.csv"


def from_edges(i: torch.Tensor, j: torch.Tensor, n: int, device: torch.device) -> torch.Tensor:
    """Symmetric 0/1 adjacency (N, N) from edge lists i, j (no self-loops)."""
    A = torch.zeros(n, n, device=device)
    A[i, j] = 1.0
    A[j, i] = 1.0
    A.fill_diagonal_(0.0)
    return A


def load_workplace(device: torch.device, min_windows: int = 1) -> torch.Tensor:
    """The 92 workers; an edge joins two people seen face to face in at least `min_windows` 20-second windows."""
    rows = [line.split(",") for line in DATA.read_text().splitlines()[1:]]
    e = torch.tensor([[int(a), int(b), int(w)] for a, b, w in rows])
    keep = e[:, 2] >= min_windows
    n = int(e[:, :2].max()) + 1
    return from_edges(e[keep, 0], e[keep, 1], n, device)


def erdos_renyi(n: int, mean_degree: float, gen: torch.Generator, device: torch.device) -> torch.Tensor:
    """G(N, p) with p = <k> / (N − 1): one coin flip per pair (upper triangle), mirrored."""
    p = mean_degree / (n - 1)
    upper = torch.triu(torch.rand(n, n, generator=gen) < p, diagonal=1).float()
    return (upper + upper.T).to(device)


def barabasi_albert(n: int, m: int, gen: torch.Generator, device: torch.device) -> torch.Tensor:
    """Preferential attachment: start from m + 1 fully linked nodes; each new node links to m distinct old nodes
    chosen with probability ∝ their degree. Growth is sequential by definition, so this is a loop over new nodes."""
    A = torch.zeros(n, n)
    A[: m + 1, : m + 1] = 1.0 - torch.eye(m + 1)
    deg = A.sum(1)
    for new in range(m + 1, n):
        targets = torch.multinomial(deg[:new], m, replacement=False, generator=gen)
        A[new, targets] = A[targets, new] = 1.0
        deg[targets] += 1
        deg[new] = m
    return A.to(device)


def complete_graph(n: int, device: torch.device) -> torch.Tensor:
    """Everybody meets everybody: the well-mixed population of the classic SIR model."""
    return torch.ones(n, n, device=device) - torch.eye(n, device=device)


def ring_lattice(n: int, k: int, device: torch.device) -> torch.Tensor:
    """N nodes on a circle, each linked to its k/2 nearest neighbours on either side: every degree is k."""
    idx = torch.arange(n, device=device)
    gap = (idx[:, None] - idx[None, :]).abs()
    gap = torch.minimum(gap, n - gap)  # distance round the ring
    return ((gap >= 1) & (gap <= k // 2)).float()


def layout(A: torch.Tensor, n_iter: int = 300, seed: int = 0) -> torch.Tensor:
    """2D positions (N, 2) for drawing, by a force-directed layout (Fruchterman–Reingold) on all pairs at once:
    linked nodes attract like springs, every pair repels."""
    gen = torch.Generator().manual_seed(seed)
    n = A.shape[0]
    x = torch.randn(n, 2, generator=gen, dtype=torch.float64)
    A = A.cpu().double()
    k = 1.0 / n**0.5
    for it in range(n_iter):
        d = x[:, None, :] - x[None, :, :]  # (N, N, 2) separation vectors
        r = d.norm(dim=-1).clamp_min(1e-3)  # (N, N)
        force = (k**2 / r**2 - A * r / k)[..., None] * d  # repulsion − attraction along d
        step = force.sum(1)
        temp = 0.1 * (1 - it / n_iter) + 1e-3
        x = x + temp * step / step.norm(dim=-1, keepdim=True).clamp_min(1e-9)
    x = x - x.mean(0)
    return (x / x.abs().max()).float()


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/others/network_sir/solve.py
"""Run many stochastic epidemics at once, the mean-field ODE, and the vaccination strategies."""

import torch

from problems._common.integrators import integrate
from problems._common.progress import steps

from . import equation

S, I, R = 0, 1, 2  # noqa: E741  state codes, the epidemiologists' letters


def simulate(A, beta, gamma, dt, n_steps, infected0, vaccinated, gen, sis=False, record=False):
    """B stochastic SIR (or SIS) epidemics on the network A at once, realisation b with rate β_b.

    infected0 (B, N) bool: the first cases; vaccinated (B, N) bool or None: immune from the start (counted in R).
    Returns counts (T, B, 3) of S, I, R; ever (T, B) people infected so far (vaccinated excluded); and, if record,
    the state of every person in realisation 0, states (T, N) int.
    """
    b, n = infected0.shape
    vacc = torch.zeros_like(infected0) if vaccinated is None else vaccinated
    state = torch.full((b, n), S, dtype=torch.long, device=A.device)
    state[vacc] = R
    state[infected0 & ~vacc] = I
    p_rec = 1.0 - torch.exp(torch.tensor(-gamma * dt))
    ever = state == I
    counts, evers, states = [], [], []

    def log():
        counts.append(torch.stack([(state == c).sum(1) for c in (S, I, R)], dim=-1))  # (B, 3)
        evers.append(ever.sum(1))
        if record:
            states.append(state[0].clone())

    log()
    for _ in steps(n_steps, "epidemic steps"):
        if not (state == I).any():
            break  # every realisation has burned out
        infected = state == I
        catch = torch.rand(b, n, generator=gen, device=A.device) < equation.infection_prob(
            beta, dt, equation.pressure(A, infected)
        )
        recover = torch.rand(b, n, generator=gen, device=A.device) < p_rec
        new_inf = (state == S) & catch
        new_rec = infected & recover
        state = torch.where(new_inf, I, state)
        state = torch.where(new_rec, S if sis else R, state)
        ever = ever | new_inf
        log()
    while len(counts) < n_steps + 1:  # pad burned-out runs so every history has the same length
        log()
    out = {"counts": torch.stack(counts), "ever": torch.stack(evers)}
    if record:
        out["states"] = torch.stack(states)
    return out


def mean_field(A, beta, gamma, x0, dt, n_steps):
    """Individual-based mean field with RK4. x0 (B, N) initial infection probabilities.  → (T, 3, B, N) = (s, x, r)."""
    y0 = torch.stack([1.0 - x0, x0, torch.zeros_like(x0)])
    return integrate(lambda y, t: equation.mean_field_rhs(y, A, beta.to(x0.dtype), gamma), y0, dt, n_steps)


def vaccinate(A, fraction, strategy, b, gen):
    """(B, N) bool: who is vaccinated in each realisation. 'targeted' takes the highest-degree people (the hubs),
    'random' a fresh random set of the same size for every realisation."""
    n = A.shape[0]
    n_vacc = int(round(fraction * n))
    if strategy == "targeted":
        rank = torch.argsort(torch.argsort(A.sum(1), descending=True, stable=True))  # 0 = biggest hub
        return (rank < n_vacc)[None, :].expand(b, n).clone()
    rank = torch.argsort(torch.argsort(torch.rand(b, n, generator=gen), dim=1), dim=1).to(A.device)
    return rank < n_vacc


def remove(A, gone):
    """The network with the people in `gone` (N,) bool removed (their rows and columns zeroed)."""
    keep = (~gone).to(A.dtype)
    return A * keep[:, None] * keep[None, :]


def threshold_sweep(A, ratios, gamma, dt, n_steps, n_runs, gen):
    """Final size against β/γ: B = len(ratios) × n_runs realisations at once, one initial case each.

    Returns the mean fraction ever infected per ratio (R,), and the mean-field final size per ratio (R,).
    """
    n = A.shape[0]
    beta = (ratios * gamma).repeat_interleave(n_runs)  # (R · runs,)
    seeds = torch.randint(n, (beta.shape[0],), generator=gen)
    infected0 = torch.zeros(beta.shape[0], n, dtype=torch.bool, device=A.device)
    infected0[torch.arange(beta.shape[0]), seeds.to(A.device)] = True
    ever = simulate(A, beta, gamma, dt, n_steps, infected0, None, gen)["ever"][-1].float() / n
    sim = ever.view(len(ratios), n_runs).mean(1)
    mf = mean_field(A, ratios * gamma, gamma, torch.full((len(ratios), n), 1.0 / n, dtype=torch.float64), dt, n_steps)
    return sim, mf[-1, 2].mean(-1)


def vaccination_sweep(A, fractions, ratio, gamma, dt, n_steps, n_runs, gen):
    """Mean fraction infected (among everybody) against the vaccinated fraction, for both strategies.

    Also the mean-field herd-immunity test: β/γ · λ_max of the network left after vaccination (< 1 ⇒ no epidemic).
    """
    n = A.shape[0]
    out = {}
    for strategy in ("targeted", "random"):
        vacc = torch.cat([vaccinate(A, f, strategy, n_runs, gen) for f in fractions.tolist()])  # (F · runs, N)
        free = torch.argsort(torch.rand(vacc.shape, generator=gen).to(A.device) + vacc.float(), dim=1)[:, :3]
        infected0 = torch.zeros_like(vacc)
        infected0.scatter_(1, free, True)  # three unvaccinated first cases per run
        beta = torch.full((vacc.shape[0],), ratio * gamma, device=A.device)
        ever = simulate(A, beta, gamma, dt, n_steps, infected0, vacc, gen)["ever"][-1].float() / n
        lam = equation.lambda_max(torch.stack([remove(A, v) for v in vacc[::n_runs]]))  # (F,) one network each
        out[strategy] = (ever.view(len(fractions), n_runs).mean(1), ratio * lam)
    return out


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/others/network_sir/plot.py
"""Animate the epidemic spreading on the contact network beside its S, I, R curves; below, the threshold and
the vaccination comparison."""

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import LineCollection

from problems._common.plotting import ACCENT, frame_indices, save_gif

COLOURS = np.array(["#93c5fd", "#dc2626", "#9ca3af"])  # S, I, R


def animate(A, pos, sim, mf, lam, lam_nb, ratio, ratios, sweep_sim, sweep_mf, fractions, vacc, vacc_beta, dt, path):
    states = sim["states"].cpu().numpy()  # (T, N)
    counts = sim["counts"].cpu().numpy()  # (T, B, 3)
    n = states.shape[1]
    alive = np.nonzero(counts[:, 0, 1] > 0)[0]
    t_end = min(len(states) - 1, int(alive[-1]) + 15 if len(alive) else 10)
    t = np.arange(len(states)) * dt
    xy = pos.cpu().numpy()
    i, j = np.nonzero(np.triu(A.cpu().numpy(), 1))
    mf = mf.cpu().numpy()

    fig, axes = plt.subplots(2, 2, figsize=(9.4, 7.2))
    (ax_net, ax_cur), (ax_thr, ax_vac) = axes

    segs = np.stack([xy[i], xy[j]], axis=1)  # (E, 2, 2)
    ax_net.add_collection(LineCollection(segs, colors="#e5e7eb", linewidths=0.5, zorder=0))
    deg = A.sum(1).cpu().numpy()
    dots = ax_net.scatter(xy[:, 0], xy[:, 1], s=12 + 1.2 * deg, c=COLOURS[states[0]], edgecolors="#374151",
                          linewidths=0.4, zorder=1)  # fmt: skip
    ax_net.set_xlim(-1.1, 1.1)
    ax_net.set_ylim(-1.1, 1.1)
    ax_net.set_aspect("equal")
    ax_net.axis("off")
    for k, lab in enumerate(["susceptible", "infected", "recovered"]):
        ax_net.scatter([], [], c=COLOURS[k], edgecolors="#374151", linewidths=0.4, label=lab)
    ax_net.legend(fontsize=7, loc="lower left", ncols=3, handletextpad=0.2, columnspacing=0.8)
    title = ax_net.set_title("")

    mean = counts.mean(1) / n  # (T, 3)
    lines = []
    for k, lab in enumerate("SIR"):
        c = ["#2563eb", "#dc2626", "#6b7280"][k]
        ax_cur.plot(t[: t_end + 1], mf[: t_end + 1, k] / n, "--", color=c, lw=1)
        ax_cur.plot(t[: t_end + 1], mean[: t_end + 1, k], color=c, lw=1, alpha=0.4)
        lines.append(ax_cur.plot([], [], color=c, lw=2, label=f"{lab} (drawn run)")[0])
    ax_cur.plot([], [], "--", color="#374151", lw=1, label="mean-field ODE")
    ax_cur.plot([], [], color="#374151", lw=1, alpha=0.4, label=f"mean of {counts.shape[1]} runs")
    ax_cur.set_xlim(0, t[t_end])
    ax_cur.set_ylim(0, 1.02)
    ax_cur.set_xlabel(r"time (units of $1/\gamma$)")
    ax_cur.set_ylabel("fraction of people")
    ax_cur.legend(fontsize=6.5, loc="center right")
    ax_cur.set_title(rf"$\beta/\gamma$ = {ratio:.3f} = {ratio * lam:.1f} / $\lambda_{{max}}$", fontsize=10)
    cursor = ax_cur.axvline(0, color="#f59e0b", lw=1)

    x = ratios.cpu().numpy() * lam
    ax_thr.plot(x, sweep_sim.cpu().numpy(), "o-", color=ACCENT[0], ms=3.5, label="stochastic SIR (mean)")
    ax_thr.plot(x, sweep_mf.cpu().numpy(), "--", color=ACCENT[3], lw=1.5, label="mean-field ODE")
    ax_thr.axvline(1, color=ACCENT[1], lw=1.2, label=r"$\beta/\gamma = 1/\lambda_{max}$")
    ax_thr.axvline(lam / (lam_nb - 1), color=ACCENT[2], lw=1.2, ls=":",
                   label=r"message passing $1/(\lambda_{NB} - 1)$")  # fmt: skip
    ax_thr.set_xlabel(r"$\beta/\gamma \cdot \lambda_{max}(A)$")
    ax_thr.set_ylabel("fraction ever infected")
    ax_thr.set_title("epidemic threshold (workplace network)", fontsize=10)
    ax_thr.legend(fontsize=6.5, loc="upper left")

    f = fractions.cpu().numpy()
    for (name, res), c in zip(vacc.items(), (ACCENT[1], ACCENT[0])):
        ax_vac.plot(f, res["targeted"][0].cpu().numpy(), "o-", color=c, ms=3.5, label=f"{name}: hubs first")
        ax_vac.plot(f, res["random"][0].cpu().numpy(), "s--", color=c, ms=3, mfc="none", label=f"{name}: random")
        herd = res["targeted"][1].cpu().numpy() < 1  # mean-field herd immunity: β/γ λ_max(after) < 1
        if herd.any():
            ax_vac.axvline(f[np.argmax(herd)], color=c, lw=0.8, ls=":")
    ax_vac.set_xlabel("fraction vaccinated")
    ax_vac.set_ylabel("fraction infected")
    ax_vac.set_title(rf"vaccination, $\beta/\gamma$ = {vacc_beta:.2f} (dotted: mean-field herd immunity)", fontsize=9)
    ax_vac.legend(fontsize=6.5, loc="upper right")
    ax_vac.set_ylim(bottom=0)

    def update(k: int):
        dots.set_facecolors(COLOURS[states[k]])
        c = counts[k, 0]
        title.set_text(f"t = {t[k]:.1f}:  S {c[0]}  I {c[1]}  R {c[2]}  (N = {n})")
        for m, line in enumerate(lines):
            line.set_data(t[: k + 1], counts[: k + 1, 0, m] / n)
        cursor.set_xdata([t[k], t[k]])

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(t_end + 1, 90), path, fps=10)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'beta_x_lambda': 3.0,
    'n_runs': 200,
    'n_synthetic': 500,
    'vacc_beta': 0.4
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.others.network_sir")]:
    del sys.modules[name]
main = importlib.import_module("problems.others.network_sir.main")
problem_dir = Path("problems/others/network_sir")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
